In [1]:
import numpy as np
import scipy.stats as stats
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

### Loading Closure Data

In [2]:
COLS = {
    'Overall Satisfaction': 'overall_satisfaction',
    'Resolution Rate': 'resolution_rate',
    'SFDC Case Original - Case number': 'case_number_closed',
    'SFDC - Clase Closure Survey : Case Status': 'case_status_closed',
    'Case Type for Portal': 'case_type_for_portal', 
    'Resident Case Record Type': 'resident_case_record_type',
    'Issue Resolution': 'issue_resolution',
    'All survey comments concatenated': 'survey_comments'
}
closure_surveys = pd.read_csv('../data/gid_case_closure_survey.csv', usecols=list(COLS)).rename(columns=COLS)

C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\1189568810.py:11: DtypeWarning: Columns (7) have mixed types. Specify dtype option on import or set low_memory=False.
  closure_surveys = pd.read_csv('../data/gid_case_closure_survey.csv', usecols=list(COLS)).rename(columns=COLS)


In [3]:
closure_surveys = closure_surveys[closure_surveys['case_status_closed'] == 'Closed']
closure_surveys.drop(columns='case_status_closed', inplace=True)

closure_surveys['survey_comments'] = closure_surveys['survey_comments'].str.lower()

In [4]:
closure_surveys['resolution_rate'].value_counts(normalize=True)

resolution_rate
True     0.609322
False    0.390678
Name: proportion, dtype: float64

In [5]:
closure_surveys['issue_resolution'].value_counts(normalize=True)

issue_resolution
Yes, on my first attempt        0.536504
No                              0.390678
Yes, after multiple attempts    0.072819
Name: proportion, dtype: float64

### Loading Open Data

In [6]:
PUBLIC_SRC = 'https://seshat.datasd.org/get_it_done_reports'
YEARS = np.arange(2016, 2026+1)

In [7]:
PUB_COLS = [
    'service_request_id', 'date_requested', 'case_age_days', 'case_record_type', 'service_name', 
    'service_name_detail', 'lat', 'lng', 'street_address', 'zipcode', 'council_district', 'comm_plan_code', 
    'comm_plan_name', 'park_name', 'case_origin', 'iamfloc', 'public_description', 'status'
    ]

closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
# open_ = pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_open_datasd.csv', usecols=PUB_COLS)

C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\3276954575.py:7: DtypeWarning: Columns (7,20) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\3276954575.py:7: DtypeWarning: Columns (17) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\3276954575.py:7: DtypeWarning: Columns (13) have mixed types. Specify dtype option on import or set low_memory=False.
  closed = pd.concat([pd.read_csv(f'{PUBLIC_SRC}/get_it_done_requests_closed_{y}_datasd.csv', usecols=PUB_COLS) for y in YEARS], ignore_index=True)
C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\

In [8]:
# pub = pd.concat([closed, open_], ignore_index=True).drop_duplicates('service_request_id')
closed.drop_duplicates('service_request_id', inplace=True)
closed['date_requested'] = pd.to_datetime(closed['date_requested'])
closed['date_requested'] = closed['date_requested'].dt.tz_localize('UTC').dt.tz_convert('US/Pacific')
closed = closed[(closed['case_age_days'] >= 0) & (closed['status'] == 'Closed')]
closed.drop(columns=['status', 'case_age_days'], inplace=True)

### Loading Referral Columns

In [9]:
REF_COLS = ['Internal Case Referral Flag', 'Internal Referral Count']
REF_RENAME = {t:t.lower().replace(' ', '_') for t in REF_COLS}
REF_RENAME.update({'Case Number': 'case_number'})
rdf = pd.read_csv('../data/referral data for UCSD-2026-11-02-23-09-16.csv', encoding='latin-1', usecols=list(REF_RENAME)).rename(columns=REF_RENAME)

#### Joining the Data

In [10]:
data = closed.merge(closure_surveys, left_on='service_request_id', right_on='case_number_closed', how='left')

In [11]:
data = closed.merge(closure_surveys, left_on='service_request_id', right_on='case_number_closed', how='left')

In [12]:
data.head()

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,lat,lng,street_address,zipcode,council_district,...,case_origin,iamfloc,public_description,overall_satisfaction,resolution_rate,case_number_closed,case_type_for_portal,resident_case_record_type,issue_resolution,survey_comments
0,100916,2016-08-21 12:22:00-07:00,TSW,Other,NaN,NaN,NaN,NaN,NaN,NaN,...,Email,NaN,I was wondering when this might be addressed.\...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,102295,2016-08-29 10:01:00-07:00,TSW,Traffic Signal Issue,NaN,33.018313,-117.115255,"16873 4S Ranch Pkwy, San Diego, CA 92127, USA",NaN,NaN,...,Web,NaN,The light at the intersection on the south eas...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,103729,2016-09-06 03:40:00-07:00,Traffic Engineering,Traffic Engineering,NaN,NaN,NaN,4626 Mercury-install street light,NaN,NaN,...,Phone,NaN,"We own the property at 4626 Mercury, the stree...",NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,104040,2016-09-07 06:25:00-07:00,TSW,Pavement Maintenance,NaN,NaN,NaN,MISSION BL&REED AV,NaN,NaN,...,Phone,NaN,very dangerous dip - East side of intersection...,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,106037,2016-09-16 04:06:00-07:00,TSW,Pothole,NaN,32.897527,-117.102791,"10377-10397 Pomerado Rd, San Diego, CA 92131, USA",NaN,NaN,...,Web,NaN,Pothole and cracked asphalt in that section fo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Outputs

properly closed cases:
- `case_status_closed` (status from customer's survey) == `'Closed'`
- `issue_resolution == 'Yes, on my first attempt'`
- `resolution_rate == True`
- `overall_satisfaction >= 9.`

(target variable is `case_record_type`)

In [13]:
OUTPUT_COLS = [
    'service_request_id', 'date_requested', 'service_name', 'service_name_detail', 'lat', 'lng', 'street_address', 
    'zipcode', 'council_district', 'comm_plan_code', 'comm_plan_name', 'park_name', 'case_origin', 'iamfloc', 'public_description'
]

TARGET_COLS = ['service_request_id', 'case_record_type']

In [14]:
proper_closures = data[
   (data['issue_resolution'] == 'Yes, on my first attempt') &
   (data['resolution_rate'] == True) &
   (data['overall_satisfaction'] >= 9.)
   ]

In [15]:
# Ensuring cases have low internal referrals (if they are not present in `rdf`, they are valid)
case_ids = proper_closures.loc[proper_closures['service_request_id'].isin(rdf['case_number'].unique()), 'service_request_id'].values
rdf[(rdf['case_number'].isin(case_ids)) & (rdf['internal_referral_count'] < 3.)].shape[0] / rdf[rdf['case_number'].isin(case_ids)].shape[0]

0.9990891939366339

In [16]:
cases_to_check = rdf.loc[(rdf['case_number'].isin(case_ids)) & ~(rdf['internal_referral_count'] <= 2), 'case_number'].values
rdf[(rdf['case_number'].isin(case_ids)) & ~(rdf['internal_referral_count'] < 3.)]

,case_number,internal_case_referral_flag,internal_referral_count
925816,4247975,1,4
1008124,4346744,1,3
1149084,4521954,1,3
1200299,4588883,1,3
1263361,4667364,1,3
1428049,4877660,1,3
1561839,5043067,1,3
1624514,5120864,1,3
1656950,5161545,1,3
1691977,5205268,1,3


In [17]:
proper_closures[proper_closures['service_request_id'].isin(cases_to_check)].head()

,service_request_id,date_requested,case_record_type,service_name,service_name_detail,lat,lng,street_address,zipcode,council_district,...,case_origin,iamfloc,public_description,overall_satisfaction,resolution_rate,case_number_closed,case_type_for_portal,resident_case_record_type,issue_resolution,survey_comments
1803098,4247975,2023-05-21 23:38:00-07:00,TSW,Weed Cleanup,NaN,32.724362,-117.236013,"3300-3398 CaÃ±on St, San Diego, CA 92106, USA",NaN,2.0,...,Web,SS-006148,The sidewalk is so overgrown with brush that t...,10.0,True,4247975.0,Weed Cleanup,NaN,"Yes, on my first attempt",potholes on canon st 92106
1816917,4346744,2023-07-28 05:15:00-07:00,TSW,Tree Maintenance,INVESTIGATE,32.799858,-117.231294,"Noyes St, San Diego, CA",NaN,1.0,...,Mobile,TR-RW-0091825,"Palm trees, overrun I?m corner of niacin grand...",10.0,True,4346744.0,Tree Maintenance,NaN,"Yes, on my first attempt",there is a sign that fell down near the corner...
1827704,4588883,2024-01-18 07:31:00-08:00,ESD Complaint/Report,Trash/Recycling Collection,Collections Truck Leak,32.956738,-117.105406,"13154 Avenida Grande , San Diego, CA 92129",92129.0,5.0,...,Mobile,SS-003185,Pile of trash left in the street by city trash...,10.0,True,4588883.0,Trash/Recycling Collection,NaN,"Yes, on my first attempt","i cynically expected this to be ignored, but t..."
1926928,4877660,2024-08-09 02:47:00-07:00,TSW,Other,NaN,32.764145,-117.101856,4305 Aldine Dr,92116.0,9.0,...,Mobile,SS-002106,Please send info again for green bin replaceme...,10.0,True,4877660.0,Other,Street Division Closed Case,"Yes, on my first attempt",aldine drive west end paved!! multiple buses e...
1979169,4521954,2023-11-28 05:22:00-08:00,TSW ROW,Vegetation Encroachment,NaN,32.811806,-117.120382,"3777 Ruffin Rd, San Diego, CA 92123, USA, San ...",92123.0,7.0,...,Web,NaN,The vegitation on the west sidewalk is overgro...,10.0,True,4521954.0,Vegetation Encroachment,NaN,"Yes, on my first attempt",NaN


In [18]:
proper_closures = proper_closures[~proper_closures['service_request_id'].isin(cases_to_check)]

In [19]:
proper_closures[OUTPUT_COLS].to_csv('resolved_cases_data-2.csv', index=False)
proper_closures[TARGET_COLS].to_csv('resolved_cases_targets-2.csv', index=False)

verified mishandled cases:
- `case_status_closed` (status from customer's survey) == `'Closed'`
- `issue_resolution == 'Yes, after multiple attempts'` (ensures that target variable is the correct department/city group)
- `resolution_rate == True`
- `overall_satisfaction >= 7.`
- exclude `all comments` where user describes improper closure or repeat submissions
- flagged for internal referrals
- high count internal referrals compared to requests for the same service

(target variable is `case_record_type`)

In [20]:
data = data.merge(rdf, left_on='service_request_id', right_on='case_number', how='left').drop(columns='case_number')
data.fillna({'internal_case_referral_flag':0, 'internal_referral_count':0}, inplace=True)
data.loc[data['internal_referral_count'] > 0, 'internal_case_referral_flag'] = 1.0
data.loc[data['service_request_id'] == 4780714, 'internal_case_referral_flag'] = 0.

In [21]:
# import re

# regex for cases that were improperly resolved/closed according to user feedback
ghost_repeat_regex = (
    r'closed without|marked closed|marked as closed|said it was closed|still there|still here|'
    r'nothing was done|nothing done|never fixed|not fixed|never came|nobody came|no one came|'
    r'closed my|closed the case|closed the ticket|before the problem was solved|first report was closed|'
    r'second time|2nd time|third time|3rd time|multiple times|reported twice|submitted twice|'
    r'three times|3 times|4 times|several times|resubmit|re-submit|previous report|prior report|'
    r'first ticket|first report|2nd get it done|broken again|broke again'
)

improperly_closed_feedback = data['survey_comments'].str.contains(ghost_repeat_regex, case=False, na=False)

cases = data[
    (data['issue_resolution'] == 'Yes, after multiple attempts') &
    (data['resolution_rate'] == True) &
    (data['overall_satisfaction'] >= 7.) &
    (data['internal_case_referral_flag'] == 1.) &
    (~improperly_closed_feedback)
    ].copy()

service_90th_referral_counts = data[data['internal_case_referral_flag'] == 1.].groupby('service_name')['internal_referral_count'].quantile(0.90)

In [22]:
cases['90th_referral_counts'] = cases['service_name'].map(service_90th_referral_counts)

In [23]:
cases = cases[cases['internal_referral_count'] >= cases['90th_referral_counts']]

In [24]:
cases[OUTPUT_COLS].to_csv('mishandled_cases_data.csv', index=False)
cases[TARGET_COLS].to_csv('mishandled_cases_targets.csv', index=False)

In [25]:
pd.options.display.max_colwidth = None
cases.loc[cases['survey_comments'].notna(), 'survey_comments']

1783196                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                 

# Load Training Dataset for the Misroute Indicator
Original type different from type when report resolved

In [71]:
priv = pd.read_csv('../../category_matched_common_support.csv')

In [73]:
df = data.merge(priv, left_on='service_request_id', right_on='CASENUMBER', how='left')

# Check if properly routed cases have good feedback

In [77]:
df[df['mismatch_label'].isna()]['overall_satisfaction'].mean()

np.float64(5.745512457952549)

In [78]:
df[df['mismatch_label'].notnull()]['overall_satisfaction'].mean()

np.float64(4.7)

In [79]:
df[df['mismatch_label'].isna()]['resolution_rate'].mean()

np.float64(0.5990533742148662)

In [80]:
df[df['mismatch_label'].notnull()]['resolution_rate'].mean()

np.float64(0.45)

In [81]:
df['mismatch_group'] = np.where(
    df['mismatch_label'].eq('mismatch'),
    'Mismatch',
    'No mismatch label'
)

summary = df.groupby('mismatch_group').agg(
    case_count=('service_request_id', 'count'),
    avg_satisfaction=('overall_satisfaction', 'mean'),
    median_satisfaction=('overall_satisfaction', 'median'),
    avg_resolution_rate=('resolution_rate', 'mean'),
    satisfaction_responses=('overall_satisfaction', 'count'),
    resolution_responses=('resolution_rate', 'count')
)

display(summary)

,case_count,avg_satisfaction,median_satisfaction,avg_resolution_rate,satisfaction_responses,resolution_responses
mismatch_group,,,,,,
Mismatch,805,4.700000,5.5,0.45,20,20
No mismatch label,2850678,5.745512,9.0,0.599053,33593,33593


# Separate mismatch, matched, and unmatched (cases not in priv)

In [82]:
df = data.merge(
    priv,
    left_on='service_request_id',
    right_on='CASENUMBER',
    how='left',
    indicator=True
)

In [83]:
print(df.groupby('_merge')['mismatch_label'].value_counts(dropna=False))

C:\Users\cjjen\AppData\Local\Temp\ipykernel_29880\2424728197.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('_merge')['mismatch_label'].value_counts(dropna=False))


_merge      mismatch_label
left_only   NaN               2849893
            mismatch                0
right_only  mismatch                0
            NaN                     0
both        mismatch              805
            NaN                   785
Name: count, dtype: int64


In [84]:
matched = df[df['_merge'] == 'both'].copy()

matched['mismatch_group'] = np.where(
    matched['mismatch_label'].eq('mismatch'),
    'Mismatch',
    'No mismatch label'
)

In [85]:
matched.groupby('mismatch_group').agg(
    case_count=('service_request_id', 'count'),
    avg_satisfaction=('overall_satisfaction', 'mean'),
    avg_resolution_rate=('resolution_rate', 'mean')
)

,case_count,avg_satisfaction,avg_resolution_rate
mismatch_group,,,
Mismatch,805,4.700000,0.45
No mismatch label,785,9.380952,0.952381
